# Week 4 — Fine-Tuning a Transformer: DistilRoBERTa

**Goal (syllabus unit V):** the project's core model. Instead of learning language from our 28k essays alone (Weeks 2–3), start from a model that has already read **~160 GB of English text**, and *fine-tune* it to tell human from AI essays.

Key ideas, in the order they appear:
- **Transformer:** reads all tokens at once and uses **self-attention**. Every token builds its representation as a weighted mix of every other token, with weights learned from how relevant each one is. No recurrence, so there's no step-by-step bottleneck (unlike the BiLSTM in `07`), and long-range links cost the same as short ones.
- **Pretraining:** RoBERTa was trained on *masked language modelling*: hide 15% of the words and predict them. To do that well it has to learn grammar, word meaning and style.
- **Distillation:** **DistilRoBERTa** is a 6-layer student trained to imitate the 12-layer RoBERTa-base teacher. It's ~2× faster and fits on our 4 GB RTX 3050, at a small cost in accuracy.
- **Fine-tuning:** put a small classification head on top and train the *whole* network on our labels with a tiny learning rate, so the pretrained knowledge is adjusted rather than overwritten.
- **Comparison, feature extraction:** freeze the pretrained layers and train only the head. This shows how much of the result comes from adapting the transformer vs just reading its existing features.

Scored like every model so far, with `evaluate` and `evaluate_on_topic`.

In [1]:
import os
import sys
sys.path.append('..')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # less wasted GPU memory; must be set before torch loads

import gc
import time
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score

from src.data import load_splits
from src.evaluate import evaluate, evaluate_on_topic, COLUMNS

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
pd.set_option('display.width', 200)
MODEL_NAME = 'distilroberta-base'
device = torch.device('cuda')
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch 2.12.0+cu130 | transformers 5.18.0
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | 4.0 GB


## Step 1 — Subword tokenization

Word-level vocabularies (Weeks 2–3) map unseen words and typos to `<unk>`. RoBERTa uses **byte-level BPE** (Byte-Pair Encoding): start from raw bytes and repeatedly merge the most frequent adjacent pairs into new tokens, up to a 50,265-token vocabulary. Consequences:
- **No unknown tokens, ever:** a typo like "becuase" becomes `bec` + `u` + `ase`. The model still sees it, and (unlike Week 3) the misspelling itself is visible.
- **Case and punctuation are kept**, and a leading space is part of the token (shown as `Ġ`).
- **Every character matters,** including *which* apostrophe was typed. A curly `’` becomes two byte tokens that a straight `'` never produces. That's a possible shortcut, tested in Step 7.

Special tokens: `<s>` at the start (its final vector is used for classification) and `</s>` at the end. The **attention mask** marks real tokens (1) vs padding (0).

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
example = "He didn’t go becuase it rained. He didn't go because it rained."
print(tokenizer.convert_ids_to_tokens(tokenizer(example)['input_ids']))
print("vocabulary size:", tokenizer.vocab_size)

['<s>', 'He', 'Ġdidn', 'âĢ', 'Ļ', 't', 'Ġgo', 'Ġbec', 'u', 'ase', 'Ġit', 'Ġr', 'ained', '.', 'ĠHe', 'Ġdidn', "'t", 'Ġgo', 'Ġbecause', 'Ġit', 'Ġr', 'ained', '.', '</s>']
vocabulary size: 50265


Tokenize every essay once. **Maximum length 512 tokens** is the model's limit (its position embeddings stop at 512). Longer essays are truncated to their first 512 tokens.

In [3]:
MAX_LEN = 512
data = load_splits()
t0 = time.time()
data['ids'] = tokenizer(data['text'].tolist(), truncation=True, max_length=MAX_LEN)['input_ids']
full_len = pd.Series([len(x) for x in tokenizer(data['text'].sample(min(3000, len(data)), random_state=SEED).tolist())['input_ids']])
print(f"tokenized {len(data):,} essays in {time.time() - t0:.0f}s")
print(f"tokens per essay (before truncation): median {full_len.median():.0f}, 90th pct {full_len.quantile(.9):.0f}")
print(f"essays that fit in {MAX_LEN} tokens: {(full_len <= MAX_LEN).mean() * 100:.0f}%")

train, val, test = (data[data['split'] == s] for s in ['train', 'val', 'test'])
ho_prompt, ho_gen = data[data['split'] == 'heldout_prompt'], data[data['split'] == 'heldout_generator']

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


tokenized 44,864 essays in 15s
tokens per essay (before truncation): median 424, 90th pct 691
essays that fit in 512 tokens: 72%


## Step 2 — The model

`AutoModelForSequenceClassification` loads the pretrained network and adds a classification head. The **load report** says what happened to each part:
- **UNEXPECTED `lm_head.*`**: the pretraining head that predicted masked words. Not needed for classification, so it's dropped.
- **MISSING `classifier.*`**: the new head (a small dense layer + output layer on the `<s>` token's final vector). It has no pretrained weights, so it starts random, and that's what we train.

`num_labels=1`: the head outputs a single logit, used with binary cross-entropy exactly as in `07`.

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
transformers.logging.set_verbosity_error()        # the report has been read; silence later loads
n_all = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.classifier.parameters())
print(f"parameters: {n_all / 1e6:.1f}M total, of which {n_head / 1e6:.2f}M in the new classifier head")
print(f"layers: {model.config.num_hidden_layers} transformer blocks, {model.config.num_attention_heads} attention heads, "
      f"hidden size {model.config.hidden_size}")
del model

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


parameters: 82.1M total, of which 0.59M in the new classifier head
layers: 6 transformer blocks, 12 attention heads, hidden size 768


82M parameters is ~50× the BiLSTM in `07`, but almost all of them arrive already trained.

## Step 3 — Batching (same idea as `07`)

Bucket essays by length so each batch of 16 needs little padding. Shuffle the order of the batches each epoch. The attention mask tells the transformer to ignore padding (`<pad>` is token id 1 for RoBERTa).

In [5]:
PAD_ID = tokenizer.pad_token_id

def batches(df, batch_size=16, shuffle=False, seed=0):
    ids, labels = df['ids'].tolist(), df['label'].to_numpy()
    order = np.argsort([len(x) for x in ids], kind='stable')
    chunks = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    if shuffle:
        np.random.default_rng(seed).shuffle(chunks)
    for chunk in chunks:
        width = max(len(ids[i]) for i in chunk)
        x = torch.full((len(chunk), width), PAD_ID, dtype=torch.long)
        for row, i in enumerate(chunk):
            x[row, :len(ids[i])] = torch.tensor(ids[i])
        yield x, (x != PAD_ID).long(), torch.tensor(labels[chunk], dtype=torch.float32), chunk

def predict(model, df, batch_size=64):
    model.eval()
    probs = np.empty(len(df), dtype=np.float32)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x, mask, _, chunk in batches(df, batch_size):
            logits = model(input_ids=x.to(device), attention_mask=mask.to(device)).logits.squeeze(1)
            probs[chunk] = torch.sigmoid(logits.float()).cpu().numpy()
    return probs

## Step 4 — Sanity check: memorise one batch

As in `07`: fine-tune on a single batch of 16 essays. Loss should fall towards 0 within a few dozen steps. If not, something is wired wrong.

**Mixed precision (fp16):** inside `torch.autocast`, most maths runs in 16-bit floats, which the RTX 3050's tensor cores do ~2× faster with half the memory. A `GradScaler` multiplies the loss before backprop so tiny fp16 gradients don't round to zero, then un-scales them before the update.

In [6]:
torch.manual_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=5e-5)
scaler = torch.amp.GradScaler()
x, mask, y, _ = next(batches(train.sample(16, random_state=SEED)))
x, mask, y = x.to(device), mask.to(device), y.to(device)
model.train()
for step in range(31):
    with torch.autocast('cuda', dtype=torch.float16):
        logits = model(input_ids=x, attention_mask=mask).logits.squeeze(1)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), y)
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    if step % 10 == 0:
        print(f"step {step:2d}  loss {loss.item():.4f}")
assert loss.item() < 0.05, "could not overfit one batch"
print("Sanity check passed.")

# Free the GPU completely before real training. `del model` alone is not enough: `loss` still points back
# through the computation graph to every parameter and its gradient (~700 MB on a 3.7 GB card).
del model, opt, scaler, logits, loss, x, mask, y
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still allocated: {torch.cuda.memory_allocated() / 1e6:.0f} MB")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

step  0  loss 0.7190


step 10  loss 0.0940


step 20  loss 0.0060


step 30  loss 0.0019
Sanity check passed.


GPU memory still allocated: 17 MB


## Step 5 — Fine-tuning recipe

The standard recipe for fine-tuning BERT-family models, with the reasons:
- **AdamW, learning rate 2e-5:** ~50× smaller than in `07`. The pretrained weights are already good, and big steps would destroy them (*catastrophic forgetting*). AdamW is Adam with *decoupled* weight decay (0.01), a mild pull of weights towards zero.
- **Linear warm-up then linear decay:** the learning rate rises from 0 over the first 6% of steps, then falls to 0 by the end. At the start the classifier head is random and its gradients are noisy; warm-up stops those early noisy updates from damaging the pretrained layers.
- **2 epochs:** fine-tuning converges fast; more epochs mostly overfit.
- **Gradient clipping at 1.0**, **fp16**, **best epoch by val AUC**, as in `07`. The best weights are copied to CPU RAM: a GPU copy would cost another 330 MB of the 3.7 GB card.

`train_model(freeze_encoder=True)` is the **feature-extraction** variant: the 6 transformer layers are frozen (`requires_grad = False`) and only the head learns. It needs a larger learning rate (1e-3), because the head starts from scratch.

In [7]:
def train_model(freeze_encoder=False, epochs=2, lr=2e-5, batch_size=16, seed=SEED):
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)
    if freeze_encoder:
        for p in model.roberta.parameters():
            p.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(train) / batch_size))
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.06 * steps), num_training_steps=steps)
    scaler = torch.amp.GradScaler()
    best_auc, best_state, log = -1, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        t0, losses = time.time(), []
        for x, mask, y, _ in batches(train, batch_size, shuffle=True, seed=seed + epoch):
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(input_ids=x.to(device), attention_mask=mask.to(device)).logits.squeeze(1)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), y.to(device))
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        val_auc = roc_auc_score(val['label'], predict(model, val))
        log.append({'epoch': epoch, 'train_loss': np.mean(losses), 'val_auc': val_auc,
                    'minutes': (time.time() - t0) / 60})
        print(f"  epoch {epoch}: train loss {log[-1]['train_loss']:.4f} | val AUC {val_auc:.5f} | {log[-1]['minutes']:.1f} min")
        if val_auc > best_auc:   # keep the best weights in CPU RAM, not on the small GPU
            best_auc, best_state = val_auc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, pd.DataFrame(log)

Train both variants (~17 min for full fine-tuning, less for the frozen one).

In [8]:
trained, logs = {}, {}
print("DistilRoBERTa, fine-tuned")
trained['DistilRoBERTa (fine-tuned)'], logs['DistilRoBERTa (fine-tuned)'] = train_model()
gc.collect(); torch.cuda.empty_cache()     # release the first model's optimizer state before training the second
print("DistilRoBERTa, frozen encoder (head only)")
trained['DistilRoBERTa (frozen, head only)'], logs['DistilRoBERTa (frozen, head only)'] = train_model(freeze_encoder=True, lr=1e-3)
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
pd.concat(logs, names=['model']).round(5)

DistilRoBERTa, fine-tuned


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

  epoch 1: train loss 0.0817 | val AUC 0.99943 | 7.7 min


  epoch 2: train loss 0.0106 | val AUC 0.99972 | 7.6 min


DistilRoBERTa, frozen encoder (head only)


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

  epoch 1: train loss 0.1180 | val AUC 0.99725 | 2.3 min


  epoch 2: train loss 0.0430 | val AUC 0.99742 | 2.3 min


peak GPU memory: 3.20 GB


epoch  train_loss  val_auc  minutes
model                                                                   
DistilRoBERTa (fine-tuned)        0      1     0.08172  0.99943  7.70702
                                  1      2     0.01057  0.99972  7.64322
DistilRoBERTa (frozen, head only) 0      1     0.11803  0.99725  2.32302
                                  1      2     0.04296  0.99742  2.32097

## Step 6 — Score on the final sets

In [9]:
probs = {}
for name, m in trained.items():
    t0 = time.time()
    probs[name] = predict(m, data)
    print(f"{name}: scored {len(data):,} essays in {time.time() - t0:.0f}s")

rows_of = lambda d: data.index.get_indexer(d.index)
sets = (test, ho_prompt, ho_gen)
all_rows = pd.DataFrame([evaluate(lambda d, p=p: p[rows_of(d)], 0.5, n, *sets) for n, p in probs.items()])
fair_rows = pd.DataFrame([evaluate_on_topic(lambda d, p=p: p[rows_of(d)], 0.5, n, *sets) for n, p in probs.items()])

key = ['test_f1', 'test_auc', 'heldout_prompt_f1', 'heldout_prompt_auc', 'heldout_gen_detection', 'heldout_gen_auc']
prev_all = pd.concat([pd.read_csv('../results/04_baseline.csv', index_col=0).loc[['TF-IDF + LinearSVM (C=1)']],
                      pd.read_csv('../results/07_bilstm.csv', index_col=0).query("subset == 'all'").drop(columns='subset')])
prev_fair = pd.concat([pd.read_csv('../results/06_on_topic.csv', index_col=0).loc[['TF-IDF + LinearSVM (C=1)']],
                       pd.read_csv('../results/07_bilstm.csv', index_col=0).query("subset == 'on_topic'").drop(columns='subset')])
compare = pd.concat({'all essays': pd.concat([prev_all, all_rows])[key],
                     'on-topic only': pd.concat([prev_fair, fair_rows])[key]}, axis=1)
compare.round(3)

DistilRoBERTa (fine-tuned): scored 44,864 essays in 182s


DistilRoBERTa (frozen, head only): scored 44,864 essays in 183s


all essays                                                                                     on-topic only                                                \
                                     test_f1 test_auc heldout_prompt_f1 heldout_prompt_auc heldout_gen_detection heldout_gen_auc       test_f1 test_auc heldout_prompt_f1 heldout_prompt_auc   
TF-IDF + LinearSVM (C=1)               0.995    1.000             0.973              0.997                 0.909           1.000         0.994    1.000             0.965              0.997   
BiLSTM (Skip-gram init)                0.984    0.999             0.867              0.964                 0.925           0.994         0.973    0.999             0.822              0.953   
BiGRU (Skip-gram init)                 0.989    1.000             0.831              0.950                 0.964           0.998         0.974    1.000             0.775              0.934   
BiLSTM (random init)                   0.988    0.999             0.876              0.966                 0.900           0.996         0.973    0.998             0.835              0.957   
DistilRoBERTa (fine-tuned)             0.985    0.999             0.955              1.000                 0.999           1.000         0.966    1.000             0.941              1.000   
DistilRoBERTa (frozen, head only)      0.965    0.998             0.917              0.998                 0.998           0.999         0.921    0.999             0.893              0.999   

                                                                         
                                  heldout_gen_detection heldout_gen_auc  
TF-IDF + LinearSVM (C=1)                          0.880           1.000  
BiLSTM (Skip-gram init)                           0.908           0.992  
BiGRU (Skip-gram init)                            0.955           0.998  
BiLSTM (random init)                              0.870           0.994  
DistilRoBERTa (fine-tuned)                        0.999           1.000  
DistilRoBERTa (frozen, head only)                 0.999           0.999

## Step 7 — Shortcut check: curly quotes

RoBERTa sees exact characters, and `04` found curly quotes (`’ “ ”`) in ~5% of AI essays and ~0% of human ones. If the model learned "curly quote → AI", that's a meaningless shortcut. Test it with **counterfactual edits**: change *only* the quote characters and see whether the prediction moves.
- AI essays (val) that contain curly quotes → replace them with straight ones. A big drop in P(AI) means it was leaning on the quotes.
- Human essays (val) that contain straight apostrophes/quotes → make them curly. A big rise means it treats curly quotes as evidence of AI.

In [10]:
to_straight = str.maketrans({'’': "'", '‘': "'", '“': '"', '”': '"'})
to_curly = str.maketrans({"'": '’', '"': '”'})
ft = trained['DistilRoBERTa (fine-tuned)']

def rescore(df, mapping):
    edited = df.assign(ids=tokenizer(df['text'].str.translate(mapping).tolist(), truncation=True, max_length=MAX_LEN)['input_ids'])
    return predict(ft, df), predict(ft, edited)

ai_curly = val[(val['label'] == 1) & val['text'].str.contains('[’‘“”]')]
hu_straight = val[(val['label'] == 0) & val['text'].str.contains('[\'"]')]
for title, df, mapping in [(f"AI essays with curly quotes -> straight ({len(ai_curly)})", ai_curly, to_straight),
                           (f"Human essays with straight quotes -> curly ({len(hu_straight)})", hu_straight, to_curly)]:
    if len(df) == 0:
        print(title, "- no essays, skipped"); continue
    before, after = rescore(df, mapping)
    print(title)
    print(f"   mean P(AI) {before.mean():.3f} -> {after.mean():.3f} | "
          f"flagged AI {(before >= .5).mean() * 100:.1f}% -> {(after >= .5).mean() * 100:.1f}% | "
          f"largest change {np.abs(after - before).max():.3f}")

AI essays with curly quotes -> straight (82)
   mean P(AI) 1.000 -> 1.000 | flagged AI 100.0% -> 100.0% | largest change 0.002


Human essays with straight quotes -> curly (1986)
   mean P(AI) 0.021 -> 0.021 | flagged AI 2.2% -> 2.2% | largest change 0.675


## Step 8 — Where does it fail?

In [11]:
p = probs['DistilRoBERTa (fine-tuned)']
for name, d in [('TEST', test), ('HELD-OUT GENERATORS', ho_gen), ('HELD-OUT PROMPTS', ho_prompt)]:
    out = d.assign(wrong=(p[rows_of(d)] >= 0.5) != d['label'])
    print(name, "- % wrong")
    print(out.groupby(['label', 'source'] if name != 'HELD-OUT PROMPTS' else ['prompt_name', 'label'])['wrong']
          .mean().mul(100).round(1).to_string(), "\n")

t = test.assign(p_ai=p[rows_of(test)])
for title, rows in [("HUMAN essays scored most AI-like", t[t['label'] == 0].nlargest(2, 'p_ai')),
                    ("AI essays scored most human-like", t[t['label'] == 1].nsmallest(2, 'p_ai'))]:
    print(f"===== {title} =====")
    for _, r in rows.iterrows():
        print(f"[p_ai={r['p_ai']:.3f} | {r['source']} | {r['prompt_name']} | on_topic={r['on_topic']}]")
        print(r['text'][:350].replace('\n', ' '), '...\n')

TEST - % wrong
label  source                            
0      persuade_corpus                       1.4
       train_essays                          0.0
1      NousResearch/Llama-2-7b-chat-hf       0.0
       chat_gpt_moth                         0.0
       cohere-command                        0.0
       kingki19_palm                         0.0
       llama2_chat                           0.0
       llama_70b_v1                          1.0
       mistral7binstruct_v1                  0.0
       mistral7binstruct_v2                  2.2
       mistralai/Mistral-7B-Instruct-v0.1    0.0
       palm-text-bison1                      0.0
       radek_500                             0.0
       radekgpt4                             0.0 

HELD-OUT GENERATORS - % wrong
label  source           
1      darragh_claude_v6    0.2
       darragh_claude_v7    0.0
       falcon_180b_v1       0.1 

HELD-OUT PROMPTS - % wrong
prompt_name                       label
Does the electoral college work?  0

## Step 9 — Save

Results to `results/08_bert.csv` (with a `subset` column, like `07`). The fine-tuned model goes to the git-ignored `models/` folder in HuggingFace format (`save_pretrained`, ~330 MB), so Weeks 6–8 (paraphrase attack, explainability, demo) can load it without retraining.

In [12]:
out = pd.concat([all_rows.assign(subset='all'), fair_rows.assign(subset='on_topic')])
out[['subset'] + COLUMNS].round(4).to_csv('../results/08_bert.csv')
ft.save_pretrained('../models/distilroberta_finetuned')
tokenizer.save_pretrained('../models/distilroberta_finetuned')
print(open('../results/08_bert.csv').read())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,subset,test_acc,test_precision,test_recall,test_f1,test_auc,heldout_prompt_f1,heldout_prompt_auc,heldout_gen_detection,heldout_gen_auc
DistilRoBERTa (fine-tuned),all,0.9894,0.9749,0.995,0.9849,0.9995,0.9549,0.9997,0.9989,0.9996
"DistilRoBERTa (frozen, head only)",all,0.9754,0.9391,0.9934,0.9655,0.9982,0.9172,0.9982,0.9982,0.9991
DistilRoBERTa (fine-tuned),on_topic,0.988,0.9382,0.9958,0.9662,0.9998,0.9411,0.9998,0.999,0.9996
"DistilRoBERTa (frozen, head only)",on_topic,0.9705,0.859,0.9915,0.9205,0.9987,0.8929,0.999,0.9985,0.999



## Key takeaways

- **Fine-tuned DistilRoBERTa is the best ranker so far, by a clear margin where it matters.** On-topic held-out prompts: AUC **0.9998** (TF-IDF 0.997, best RNN 0.957). Held-out generators: **99.9% detected** (Claude v6/v7, Falcon-180B never seen in training), vs BiGRU 95.5% and TF-IDF 88.0%. Pretraining on general English carries over to unseen models and topics far better than anything trained on our essays alone.
- **But it over-flags humans at the default 0.5 threshold.** On-topic test precision 0.938 vs TF-IDF ~1.0; 1.4% of test Persuade essays flagged (TF-IDF ~0.1%); and **10.5% of human "Summer projects" essays** flagged on the unseen prompt. The ranking is near-perfect (AUC ≈ 1), so this is the **threshold calibration** problem from `04`, more pronounced here. Next step: pick the threshold on `val` for a target false-positive rate (e.g. ≤ 1%) instead of using 0.5. For an accusation tool, false positives are the costly error.
- **Frozen encoder (head only) already ranks almost perfectly** (held-out-prompt AUC 0.999, generators 99.8%), training 3× faster (2.3 vs 7.7 min/epoch). Most of the power is in the pretrained features. Fine-tuning mainly improves the decision boundary: on-topic test F1 0.921 → 0.966.
- **Curly-quote shortcut: not used.** Straightening quotes in 82 AI essays changed nothing (all still flagged); curling quotes in 1,986 human essays left the flag rate at 2.2%. One essay moved by 0.68, so the model isn't fully immune, but it isn't a systematic shortcut.
- **Failure cases repeat across every model so far:** fluent, formal *human* essays score as AI (the fairness risk), and a few `mistral7binstruct_v2` "AI" essays read like learner English ("The grow is the stage of life…"). Every model misses those, which supports the label-noise explanation.
- **Practical:** 82M parameters, 512 tokens, batch 16, fp16; peak 3.2 GB of the 3.7 GB usable; ~15 min to fine-tune, 3 min to score 44,864 essays. The model is saved to `models/distilroberta_finetuned` for the explainability, paraphrase-attack and demo weeks.
